In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

# Load dataset
data_path = '/content/telecom_churn200.csv'
df = pd.read_csv(data_path)

print("Dataset Shape:", df.shape)
display(df.head())


In [ ]:
# Identify target and features
target_col = 'churn' if 'churn' in df.columns else [col for col in df.columns if 'churn' in col.lower()][0]
print(f"Target column identified: {target_col}")

X = df.drop(columns=[target_col])
y = df[target_col]

# Handle potential boolean or string target conversion to integer
if y.dtype == 'object' or y.dtype == 'bool':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)

# Identify numerical and categorical columns
# Filter out high-cardinality non-predictive identifier columns (e.g., ID, name, phone)
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
potential_cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

cat_cols = []
for col in potential_cat_cols:
    # Exclude columns where almost every row has a unique value (like IDs)
    if X[col].nunique() < 100:
        cat_cols.append(col)
    else:
        print(f"Skipping high-cardinality non-predictive column: {col}")

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

In [ ]:
# Build preprocessing pipelines
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

# Partition dataset into 80% train and 20% test with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")


In [ ]:
# Define the three models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

# Train and evaluate each model using the pipeline
for name, model in models.items():
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    # Train
    clf.fit(X_train, y_train)

    # Predict
    y_pred = clf.predict(X_test)

    print(f"\n{'='*10} {name} Evaluation {'='*10}")
    print("Accuracy:", accuracy_score(y_test, y_pred))

    try:
        if hasattr(clf, "predict_proba"):
            y_prob = clf.predict_proba(X_test)[:, 1]
            print("ROC AUC:", roc_auc_score(y_test, y_prob))
    except Exception as e:
        print(f"Could not calculate ROC AUC: {e}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))